# 01. Intro to RAG with Pinecone

> *If you ask a language model about something it's never seen, it'll either make it up or shrug. RAG is the boring, practical fix: pull the right passage out of a search index, paste it in front of the question, and let the model do its thing.*

This is the first of three notebooks. Over the series we'll build a small Q&A bot that answers questions about Pinecone by searching Pinecone's own docs. Meta, but it works.

**This first notebook is the crawl.** We're not embedding anything, we're not calling an LLM, and we're not loading a real corpus. All we're doing is getting five tiny vectors into Pinecone and one query back out. Just enough to see the shape of the thing.

## What we're doing here

Three goals, nothing more:

1. Get a Pinecone client talking to a serverless index.
2. Put five vectors in.
3. Ask "what's closest to this one?" and see sensible results come back.

Everything fancier shows up later. NB2 pushes on the API surface. NB3 plugs in a real model.

## The pipeline we're heading toward

```
 ┌─────────┐   ┌─────────┐   ┌──────────┐   ┌──────────┐
 │  Query  │──▶│  Embed  │──▶│ Retrieve │──▶│ Generate │──▶ Answer
 └─────────┘   └─────────┘   └──────────┘   └──────────┘
                                   ▲
                                   │
                         ┌──────────────────┐
                         │   Vector DB      │
                         │   (Pinecone)     │
                         │   your docs here │
                         └──────────────────┘
```

Every RAG system is some version of these four boxes. Today we're only looking at `Retrieve`, and specifically the storage side of it. The rest lands in later notebooks.

## Why we need a vector DB at all

Normal databases look for exact matches. Find rows where `genre = 'thriller'`, find documents containing the word `serverless`. That's fine until a user asks for "scary movies" or "without managing servers", at which point the keyword match falls over even though the meaning is right there.

A vector database doesn't care about words. It stores each document as a long list of numbers (a dense vector), produced by an embedding model, where similar meanings end up near each other. Then retrieval becomes geometry: "give me the five points closest to this one." That's it. That's the whole idea.

The rest of this module is about the machinery around that one idea, making it fast and flexible at scale.

## Setup (one time)

1. Copy `.env.example` to `.env` and paste your Pinecone API key. The free tier is fine. Grab one at [app.pinecone.io](https://app.pinecone.io/).
2. `pip install -r requirements.txt`
3. Run the cell below.

If the key is missing or the file is in the wrong place, `load_env()` tells you straight away. No mysterious 401s.

In [1]:
from helpers import load_env, get_pinecone_client

cfg = load_env()
pc = get_pinecone_client(cfg)
print(f"Connected to Pinecone. Default region: {cfg.pinecone_cloud}/{cfg.pinecone_region}")

Connected to Pinecone. Default region: aws/us-east-1


## Serverless or pod-based?

Pinecone has two flavours of index. Serverless auto-scales and you pay per read, write, and stored vector. Pod-based makes you pick a size up front and is the right call for predictable, very high throughput.

For learning, prototyping, and most apps under about 10M vectors, serverless is the answer. We use it everywhere in this series. If you ever need to switch, the only thing that changes is the `spec=` argument to `create_index`.

## Create a tiny index

We're making a 4-dimensional index on purpose. Real embeddings are 1,536 dimensions, which is hard to reason about. Four dimensions is small enough that we can write the vectors by hand and *see* why the results come back the way they do.

In [2]:
from helpers import ensure_index

INDEX_NAME = "rag-unpacked-intro-demo"

index = ensure_index(
    pc,
    name=INDEX_NAME,
    dimension=4,
    metric="cosine",
    cloud=cfg.pinecone_cloud,
    region=cfg.pinecone_region,
)
print(f"Index {INDEX_NAME!r} is ready.")

Index 'rag-unpacked-intro-demo' is ready.


## Look around

Three calls you'll reach for constantly:

- `pc.list_indexes()` lists every index in your project.
- `pc.describe_index(name)` gives the metadata for one: host, dimension, metric, status.
- `index.describe_index_stats()` gives live stats (vector count, namespaces). Use it after every write while you're learning, it's a great sanity check.

In [3]:
print("--- list_indexes ---")
print(pc.list_indexes())

print("\n--- describe_index ---")
print(pc.describe_index(INDEX_NAME))

print("\n--- describe_index_stats (should be empty) ---")
print(index.describe_index_stats())

--- list_indexes ---
[{
    "name": "pinecone-datacamp",
    "metric": "cosine",
    "host": "pinecone-datacamp-9agnsww.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}, {
    "name": "rag-unpacked-intro-demo",
    "metric": "cosine",
    "host": "rag-unpacked-intro-demo-9agnsww.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mo

## Upsert: the one write call you need

"Upsert" is insert-or-update, keyed by id. New id, it inserts. Existing id, it overwrites. There's no separate `create` and `update` at the data level, it's just this.

We'll put in five vectors. They're arranged so that related ideas point in similar directions, which makes the query results predictable enough to check by eye.

| id       | vector                 | label                |
|----------|------------------------|----------------------|
| `cat`    | `[1.0, 0.0, 0.0, 0.0]` | "small furry animal" |
| `kitten` | `[0.9, 0.1, 0.0, 0.0]` | "small furry animal" |
| `tiger`  | `[0.8, 0.0, 0.2, 0.0]` | "large furry animal" |
| `car`    | `[0.0, 0.0, 1.0, 0.0]` | "vehicle"            |
| `truck`  | `[0.0, 0.0, 0.9, 0.1]` | "vehicle"            |

Nobody writes real embeddings by hand. But on four dimensions, the geometry is obvious: animals and vehicles point in different directions, and cats, kittens, and tigers cluster near each other on the "animal" side.

In [4]:
vectors = [
    {"id": "cat",    "values": [1.0, 0.0, 0.0, 0.0], "metadata": {"label": "small furry animal"}},
    {"id": "kitten", "values": [0.9, 0.1, 0.0, 0.0], "metadata": {"label": "small furry animal"}},
    {"id": "tiger",  "values": [0.8, 0.0, 0.2, 0.0], "metadata": {"label": "large furry animal"}},
    {"id": "car",    "values": [0.0, 0.0, 1.0, 0.0], "metadata": {"label": "vehicle"}},
    {"id": "truck",  "values": [0.0, 0.0, 0.9, 0.1], "metadata": {"label": "vehicle"}},
]

index.upsert(vectors=vectors)
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '179',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 23 Apr 2026 18:56:04 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '40',
                                    'x-pinecone-request-latency-ms': '40',
                                    'x-pinecone-response-duration-ms': '41'}},
 'dimension': 4,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 5}},
 'storageFullness': 0.0,
 'total_vector_count': 5,
 'vector_type': 'dense'}


## Query: the payoff

We query with `[0.95, 0.05, 0.0, 0.0]`, which is almost pure small-furry-animal with a tiny lean toward kitten-ish. So the top three should be kitten (strongest match, it leans the same way), then cat (pure animal), then tiger (partly animal). Nothing vehicle-shaped should make the list.

In [5]:
query_result = index.query(
    vector=[0.95, 0.05, 0.0, 0.0],
    top_k=3,
    include_metadata=True,
)

for match in query_result["matches"]:
    print(f"  {match['id']:8s}  score={match['score']:.4f}  {match['metadata']}")

  cat       score=0.9986  {'label': 'small furry animal'}
  kitten    score=0.9981  {'label': 'small furry animal'}
  tiger     score=0.9688  {'label': 'large furry animal'}


## What those scores actually mean

The numbers are **cosine similarity**. A perfect match is 1.0, unrelated is 0.0, opposite is -1.0. Geometrically it's just the cosine of the angle between two arrows drawn from the origin. Two vectors that point the same way score close to 1, regardless of their length.

That's why `cat` and `kitten` score almost identically in our query: the arrows are pointing in almost exactly the same direction, the difference in magnitude doesn't matter. And `car` and `truck` don't show up at all, because their arrows point in a completely different direction and the cosine is near zero.

Once that clicks, the rest of the API is just packaging around this one idea.

## Where we're going

Real embeddings aren't hand-written, they come from a model. OpenAI's `text-embedding-3-small` takes a string and returns a 1,536-dimensional vector, with the same "similar meanings point the same way" property you just watched work on four dimensions.

We'll wire that up in NB3. Before that, NB2 stays on toy vectors but pushes on everything else Pinecone lets you do: filters, updates, deletes, namespaces. Same building block, more moves.

## Cleanup

Drop the demo index so it doesn't sit in your project. The `has_index` check makes this safe to run twice.

In [6]:
if pc.has_index(INDEX_NAME):
    pc.delete_index(INDEX_NAME)
    print(f"Deleted {INDEX_NAME!r}.")
else:
    print(f"{INDEX_NAME!r} already gone.")

Deleted 'rag-unpacked-intro-demo'.


## Recap

- A vector database stores dense vectors and quickly finds the closest ones to a query.
- Serverless indexes are the default. You don't pick a size, you just use it.
- An index's dimension is fixed at creation and has to match your embedding model.
- Cosine similarity is the right metric for text, and it's the default.
- The five calls you'll use most: `create_index`, `upsert`, `query`, `describe_index_stats`, `delete_index`.

Next up: [02. Vector Manipulation](./02_vector_manipulation.ipynb). Still no models, but we'll start filtering, updating, deleting, and splitting data across namespaces.